# EEGNet from NeuroHID Runtime Features

This notebook:
1. Sets up PyTorch and uses GPU when available.
2. Connects to `neurohid-service` telemetry.
3. Collects `decision_event` feature vectors from Runtime-ML protocol v2.
4. Builds and trains an EEGNet-style classifier on those features.

Before running: ensure `neurohid-service` is running and emitting Runtime-ML telemetry.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")

In [ ]:
from neurohid_ml import NeuroHidNotebook

nh = NeuroHidNotebook(
    control_transport="tcp",
    control_host="127.0.0.1",
    control_port=47385,
)

_ = nh.ensure_connected_stream(rescan=True)
snapshot = nh.snapshot()

{
    "runtime_mode": snapshot.get("runtime_mode_state"),
    "ml_bridge_connected": snapshot.get("ml_bridge_connected"),
    "ml_bridge_stalled": snapshot.get("ml_bridge_stalled"),
    "device_connected": snapshot.get("device_connected"),
    "connected_streams": [
        s
        for s in snapshot.get("discovered_streams", [])
        if isinstance(s, dict) and s.get("connected")
    ],
}

In [ ]:
from collections import Counter
import time

TARGET_SAMPLES = 800
MAX_MESSAGES = 10_000


def action_to_label(action: dict) -> str:
    if not isinstance(action, dict):
        return "none"

    mouse = action.get("mouse")
    keyboard = action.get("keyboard")

    if isinstance(mouse, dict):
        movement = mouse.get("movement")
        if isinstance(movement, dict):
            dx = float(movement.get("dx", 0.0))
            dy = float(movement.get("dy", 0.0))
            if abs(dx) > 1e-6 or abs(dy) > 1e-6:
                return "mouse_move"

        buttons = mouse.get("buttons")
        if isinstance(buttons, list) and buttons:
            first = buttons[0] if isinstance(buttons[0], dict) else {}
            button = first.get("button", "unknown")
            pressed = first.get("pressed")
            if pressed is True:
                return f"mouse_{button}_press"
            if pressed is False:
                return f"mouse_{button}_release"
            return f"mouse_{button}"

    if isinstance(keyboard, dict):
        events = keyboard.get("events")
        if isinstance(events, list) and events:
            first = events[0] if isinstance(events[0], dict) else {}
            key = first.get("key", "unknown")
            pressed = first.get("pressed")
            if pressed is True:
                return f"key_{key}_press"
            if pressed is False:
                return f"key_{key}_release"
            return f"key_{key}"

    return "none"


start = time.time()
features = []
labels = []

for msg in nh.iter_telemetry(max_messages=MAX_MESSAGES):
    if msg.get("kind") != "decision_event":
        continue

    payload = msg.get("payload", {})
    vec = payload.get("feature_values")
    if not isinstance(vec, list) or not vec:
        continue

    try:
        features.append([float(v) for v in vec])
    except (TypeError, ValueError):
        continue

    labels.append(action_to_label(payload.get("action")))

    if len(features) >= TARGET_SAMPLES:
        break

elapsed = time.time() - start
print(f"Collected {len(features)} decision_event samples in {elapsed:.1f}s")
print("Top labels:", Counter(labels).most_common(10))

if len(features) < 32:
    raise RuntimeError(
        "Not enough samples. Increase MAX_MESSAGES or keep runtime running longer."
    )

In [ ]:
import math


def infer_channel_count(snapshot: dict, feature_len: int) -> int:
    streams = snapshot.get("discovered_streams", [])
    if isinstance(streams, list):
        for stream in streams:
            if not isinstance(stream, dict) or not stream.get("connected"):
                continue
            channel_count = stream.get("channel_count")
            if isinstance(channel_count, int) and channel_count > 0:
                if feature_len % channel_count == 0:
                    return channel_count

    for candidate in [128, 64, 32, 16, 14, 8, 4, 2, 1]:
        if feature_len % candidate == 0:
            return candidate

    return 1


feature_len = len(features[0])
if any(len(row) != feature_len for row in features):
    min_len = min(len(row) for row in features)
    features = [row[:min_len] for row in features]
    feature_len = min_len

channels = infer_channel_count(snapshot, feature_len)
time_steps = feature_len // channels
if channels * time_steps != feature_len:
    raise RuntimeError("Could not infer a valid (channels, time_steps) shape.")

label_names = sorted(set(labels))
if len(label_names) < 2:
    raise RuntimeError(f"Need >=2 labels for classification, got: {label_names}")
label_to_idx = {name: idx for idx, name in enumerate(label_names)}

x = torch.tensor(features, dtype=torch.float32)
x = (x - x.mean(dim=0, keepdim=True)) / (x.std(dim=0, keepdim=True) + 1e-6)
x = x.view(-1, 1, channels, time_steps)
y = torch.tensor([label_to_idx[name] for name in labels], dtype=torch.long)

print("Tensor shape:", tuple(x.shape), "(N, 1, C, T)")
print("Classes:", label_to_idx)

In [ ]:
class EEGNet(nn.Module):
    """Compact EEGNet-style model for (N, 1, C, T) inputs."""

    def __init__(
        self,
        channels: int,
        num_classes: int,
        temporal_kernel: int = 64,
        f1: int = 8,
        depth_multiplier: int = 2,
        separable_kernel: int = 16,
        dropout: float = 0.5,
    ) -> None:
        super().__init__()
        f2 = f1 * depth_multiplier

        self.block1 = nn.Sequential(
            nn.Conv2d(
                1,
                f1,
                kernel_size=(1, temporal_kernel),
                padding=(0, temporal_kernel // 2),
                bias=False,
            ),
            nn.BatchNorm2d(f1),
            nn.Conv2d(f1, f2, kernel_size=(channels, 1), groups=f1, bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                f2,
                f2,
                kernel_size=(1, separable_kernel),
                padding=(0, separable_kernel // 2),
                groups=f2,
                bias=False,
            ),
            nn.Conv2d(f2, f2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(f2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        return self.head(x)


model = EEGNet(channels=channels, num_classes=len(label_names)).to(device)
model

In [ ]:
torch.manual_seed(7)

dataset = TensorDataset(x, y)
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (logits.argmax(dim=1) == yb).sum().item()
        train_total += xb.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)
            val_correct += (logits.argmax(dim=1) == yb).sum().item()
            val_total += xb.size(0)

    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss / max(train_total, 1):.4f} "
        f"train_acc={train_correct / max(train_total, 1):.3f} "
        f"val_loss={val_loss / max(val_total, 1):.4f} "
        f"val_acc={val_correct / max(val_total, 1):.3f}"
    )

In [ ]:
from pathlib import Path

output_dir = Path("./artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = output_dir / "eegnet_runtime_features.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "label_to_idx": label_to_idx,
        "channels": channels,
        "time_steps": time_steps,
        "feature_len": feature_len,
    },
    checkpoint_path,
)

print(f"Saved model checkpoint to: {checkpoint_path.resolve()}")